# Final Project

Movie review sentiment analysis.

In [ ]:
from pathlib import Path
import jsonlines
import json
import itertools
import re


import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, FeatureUnion


In [ ]:
def load_data(path):
    df = pd.read_csv(path).rename(columns={k: k.lower() for k in ['ID', 'TEXT', 'LABEL']})
    return df

df = load_data('data/train.csv')
df.head()

# Just document this so you don't get it backwards latter: LABELS[int] gets sentiment.
LABELS = ['non-movie', 'postive', 'negative']

In [ ]:
df.isna().sum()
df = df.dropna()

In [ ]:
train, dev = train_test_split(df, random_state=879345)

In [ ]:
train.label.value_counts(normalize=True)

In [ ]:
text_to_vect = CountVectorizer(ngram_range=(1, 2), min_df=2, stop_words='english')
text_to_vect.fit(train.text)


## Modeling

In [ ]:


class ExpressionTransformer(BaseEstimator, TransformerMixin):
    """Perform text feature extraction by labeling inputs that match expressions.

    The tranform method returns an n_samples x n_expressions array.
    """
    def __init__(self, expressions: list[str]):
        # self.expressions = [
        #     exp if isinstance(exp, re.Pattern)
        #     else re.compile(exp, re.IGNORECASE)
        #     for exp in expressions
        # ]
        self.expressions = expressions

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        is_pandas = isinstance(X, pd.Series)
        if not is_pandas:
            X = pd.Series(X)
        transformed = []
        for expression in self.expressions:
            transformed.append(X.str.contains(expression).values)
        return np.stack(transformed).T

def make_mnb_pipeline(config):
    """Build model from config."""
    config_type = config['model_type']
    if config['model_type'] == 'multinomial_count_vect':
        vectorizer_config = config['vectorizer']
        model_config = config['classifier']
        vectorizer = CountVectorizer(**vectorizer_config)
        classifier = MultinomialNB(**model_config)
        return Pipeline([('preprocessor', vectorizer), ('classifier', classifier)])
    elif config_type == 'multinomial_count_vect_with_features':
        vectorizer_config = config['vectorizer']
        model_config = config['classifier']
        vectorizer = CountVectorizer(**vectorizer_config)
        classifier = MultinomialNB(**model_config)
        expressions = ExpressionTransformer(**config['expressions'])
        pipeline = Pipeline([
            ('features', FeatureUnion([
                ('text', vectorizer),
                ('expressions', expressions),
            ])),
            ('clf', classifier)
        ])
        return pipeline
    elif config_type == 'logistic_with_tfidf':
        vectorizer_config = config['vectorizer']
        model_config = config['classifier']
        vectorizer = TfidfVectorizer(**vectorizer_config)
        classifier = LogisticRegression(**model_config)
        expressions = ExpressionTransformer(**config['expressions'])
        pipeline = Pipeline([
            ('features', FeatureUnion([
                ('text', vectorizer),
                ('expressions', expressions),
            ])),
            ('clf', classifier)
        ])
        return pipeline
    elif config_type == 'voting':
        multinomial = make_mnb_pipeline(config['multinomial'])
        logistic = make_mnb_pipeline(config['logistic'])
        return StackingClassifier(
            [
                ('multinomial', multinomial),
                ('logistic', logistic)],
            final_estimator=LogisticRegression()
        )
    raise ValueError(f"Unsupported model type {config['model_type']}")

def score_predictions(y_true, y_hat) -> dict:
    return {
        'macro_f1': f1_score(y_true, y_hat, average='macro'),
        'confusion': confusion_matrix(y_true, y_hat).tolist()
    }

def experiment(model_config, train, test, experiments_dir='data/experiments'):
    """Run an experiment based on the model configuration."""
    existing = get_experiment_results(model_config, experiments_dir)
    if existing:
        print(f"Experiment has already been run. Try something new.")
        print({k: v for k, v in existing.items() if k != 'config'})
        return
    model = make_mnb_pipeline(model_config)
    X = train.text
    y = train.label
    model.fit(X, y)
    y_test = test.label
    X_test = test.text
    y_hat = model.predict(X_test)
    scores = score_predictions(y_test, y_hat)
    result = {'model_type': model_config['model_type'], 'config': model_config} | scores

    experiments_dir = Path(experiments_dir)
    if not experiments_dir.exists():
        experiments_dir.mkdir(parents=True)
    file_name = experiments_dir / 'experiments.jsonl'
    if not file_name.exists():
        file_name.touch()
    with jsonlines.open(file_name, 'a') as f:
        f.write(result)
    print(f"Experiment finished.\n{model_config}")
    print(f"{scores}")
    return model

def get_experiment_results(model_config, experiment_dir):
    experiment_dir = Path(experiment_dir)
    with jsonlines.open(experiment_dir / 'experiments.jsonl') as f:
        for experiment in f:
            experiment_config = experiment['config']

            if experiment_config.get('vectorizer'):
                ngram_range = experiment_config.get('vectorizer').get('ngram_range')
                if ngram_range:
                    experiment_config['vectorizer']['ngram_range'] = tuple(ngram_range)
            if experiment['config'] == model_config:
                return experiment
    return {}

def load_best_config(experiment_dir):
    experiment_dir = Path(experiment_dir)
    best = -np.inf
    best_experiment = None
    with open(experiment_dir / 'experiments.jsonl') as f:
        for experiment in f.read():
            f1 = experiment['macro_f1']
            if f1 > best:
                best = f1
                best_experiment = experiment
    return best_experiment


In [ ]:
logistic_config = {
    'model_type': 'logistic_with_tfidf',
    'vectorizer': {'ngram_range': (1, 2), 'min_df': 10, 'stop_words': 'english'},
    'classifier': {'max_iter': 200},
    'expressions': {'expressions': [r'game|player', r'book|author|reader', r'song', r'dvd']}
}
multinomial_config = {
    'model_type': 'multinomial_count_vect',
    'vectorizer': {'ngram_range': (1, 2), 'min_df': 7, 'stop_words': 'english'},
    'classifier': {},
    # 'expressions': {'expressions': [r'game|player', r'book|author|reader', r'song', r'dvd']}
}
model_config = {
    'model_type': 'voting',
    'logistic': logistic_config,
    'multinomial': multinomial_config
}

In [ ]:
# model = experiment(model_config, train[:1000], dev[:1000], 'data/test_experiment')
model = experiment(model_config, train, dev)


In [ ]:
def error_analysis(model, dev_data: pd.DataFrame):
    y_hat = model.predict(dev_data.text)
    y = dev_data.label
    results_df = dev_data.copy().assign(y_hat=y_hat)
    results_df[results_df.label != results_df.y_hat]

    errors = results_df

    # look at incorrect classificaitons at positive movie reviews
    mistake_types = [(i, j) for i, j in itertools.product(range(0, 3), repeat=2) if i != j]

    for true, pred in mistake_types:
        # if (true, pred) != (1, 2) and (true, pred) != (2, 1):
        #     continue
        mistakes = errors[(errors.label == true) & (errors.y_hat == pred)]
        print(f"{LABELS[true]} which were classed as {LABELS[pred]}")
        n_iter = 0
        for _, mistake in mistakes.iterrows():
            print('\n'.join(textwrap.wrap('*  ' + mistake.text)))
            n_iter += 1
            if n_iter == 5:
                break
        print()

error_analysis(model, dev)

In [ ]:
test_data = load_data('data/test.csv')
test_data

## Create Sumbission

In [ ]:
model.fit(df.text, df.label)

In [ ]:
submission = all_data.assign(LABEL=model.predict(test_data.text)).rename(columns={'id': 'ID'})

In [ ]:
sumbission[['ID', 'LABEL']].to_csv('submission.csv', index=False)